# 03 Sentinel-2 Acquisition

Request resumable Earth Engine Drive exports for cloud-masked Sentinel-2 composites.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from igcd.config import load_config
from igcd.ee_utils import authenticate_and_initialize
from igcd.sentinel import export_sentinel_for_inventory
from igcd.utils import setup_logging

config = load_config(PROJECT_ROOT / 'config' / 'config.json')
logger = setup_logging(config.paths['logs'])
authenticate_and_initialize(project=config.raw['earth_engine']['project'])

inventory_path = config.paths['processed'] / 'inventory' / 'glacier_inventory.geojson'
inventory = gpd.read_file(inventory_path)
max_glaciers = config.raw.get('processing', {}).get('max_glaciers_per_export_batch')
if max_glaciers is not None:
    inventory = inventory.head(int(max_glaciers)).copy()
inventory['geometry'] = inventory.geometry.apply(lambda geom: geom.__geo_interface__)
manifest = export_sentinel_for_inventory(
    inventory,
    config,
    logger,
    config.paths['reports'] / 'sentinel_acquisition_manifest.csv',
    max_tasks=config.raw['export'].get('max_tasks_per_run')
)
manifest.head()